# Dataset overview

This executed notebook presents the four dissertation datasets using
repository-safe aggregate statistics and fully synthetic option records.
Licensed OptionMetrics rows are never loaded here.

In [ ]:
from pathlib import Path
import json
import os
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display


def find_repository_root():
    for candidate in (Path.cwd(), *Path.cwd().parents):
        if (candidate / "docs" / "data" / "dataset_profile.json").is_file():
            return candidate
    raise FileNotFoundError("Run this notebook from inside the repository.")


ROOT = find_repository_root()
LANGUAGE = os.environ.get("DATASET_NOTEBOOK_LANGUAGE", "en")
if LANGUAGE not in {"en", "zh"}:
    raise ValueError("DATASET_NOTEBOOK_LANGUAGE must be 'en' or 'zh'")

LABELS = {
    "en": {
        "table": "Table",
        "rows": "Rows",
        "period": "Observation period",
        "size": "Size (MiB)",
        "columns": "Columns",
        "missing": "Overall missing cells (%)",
        "date": "Date",
        "symbol": "Symbol",
        "type": "Type",
        "strike": "Strike",
        "iv": "Synthetic IV",
        "chart_title": "Synthetic volatility smile",
        "x_axis": "Strike",
        "y_axis": "Implied volatility",
        "notice": "All displayed option contracts are synthetic.",
    },
    "zh": {
        "table": "数据表",
        "rows": "行数",
        "period": "观测区间",
        "size": "大小（MiB）",
        "columns": "列数",
        "missing": "总体缺失单元格（%）",
        "date": "日期",
        "symbol": "合约代码",
        "type": "类型",
        "strike": "执行价",
        "iv": "合成隐含波动率",
        "chart_title": "合成波动率微笑",
        "x_axis": "执行价",
        "y_axis": "隐含波动率",
        "notice": "所有展示的期权合约均为合成记录。",
    },
}
labels = LABELS[LANGUAGE]
profile = json.loads(
    (ROOT / "docs" / "data" / "dataset_profile.json").read_text(encoding="utf-8")
)
dictionary_path = ROOT / "docs" / "data" / (
    "data_dictionary_en.csv" if LANGUAGE == "en" else "data_dictionary_zh.csv"
)
synthetic_path = ROOT / "examples" / "synthetic_option_data.csv"

## Coverage and scale

The table below is generated from the aggregate-only profile.

In [ ]:
coverage_rows = []
for name, table in profile["tables"].items():
    coverage_rows.append({
        labels["table"]: name,
        labels["rows"]: table["row_count"],
        labels["period"]: (
            f'{table["date_range"]["min"]} – {table["date_range"]["max"]}'
        ),
        labels["size"]: round(table["file_size_bytes"] / (1024 ** 2), 2),
    })
coverage = pd.DataFrame(coverage_rows)
display(coverage.style.format({labels["rows"]: "{:,.0f}", labels["size"]: "{:,.2f}"}))

## Data quality

Missingness is calculated from per-column missing counts in the aggregate
profile. No source row is required.

In [ ]:
quality_rows = []
for name, table in profile["tables"].items():
    total_cells = table["row_count"] * len(table["columns"])
    missing_cells = sum(table["missing_counts"].values())
    quality_rows.append({
        labels["table"]: name,
        labels["columns"]: len(table["columns"]),
        labels["missing"]: 100 * missing_cells / total_cells if total_cells else 0,
    })
quality = pd.DataFrame(quality_rows)
display(quality.style.format({labels["missing"]: "{:.3f}"}))

## Data dictionary

The first entries illustrate how source fields are used in the dissertation.

In [ ]:
dictionary = pd.read_csv(dictionary_path)
display(dictionary.head(12))

## Synthetic option example

Every contract below is invented and marked `synthetic_only`.

In [ ]:
synthetic = pd.read_csv(synthetic_path)
if not synthetic["symbol_flag"].eq("synthetic_only").all():
    raise ValueError("Synthetic example contains an unmarked row")
synthetic_view = pd.DataFrame({
    labels["date"]: synthetic["date"],
    labels["symbol"]: synthetic["symbol"],
    labels["type"]: synthetic["cp_flag"],
    labels["strike"]: synthetic["strike_price"] / 1000,
    labels["iv"]: synthetic["impl_volatility"],
})
print(labels["notice"])
display(synthetic_view.style.format({
    labels["strike"]: "{:,.0f}",
    labels["iv"]: "{:.1%}",
}))

## Illustrative volatility smile

This figure is generated only from the synthetic example and is not an
empirical result.

In [ ]:
smile = synthetic.assign(strike=synthetic["strike_price"] / 1000).sort_values("strike")
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(
    smile["strike"],
    smile["impl_volatility"],
    color="#2563eb",
    marker="o",
    linewidth=2.5,
)
ax.set_title(labels["chart_title"])
ax.set_xlabel(labels["x_axis"])
ax.set_ylabel(labels["y_axis"])
ax.yaxis.set_major_formatter(lambda value, position: f"{value:.0%}")
ax.grid(alpha=0.25)
fig.tight_layout()
plt.show()